<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/muse_gguf_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls -ltha /content/drive/MyDrive/topogpt

total 48K
drwx------ 12 root root 4.0K Oct 30 11:18 .
drwx------ 47 root root 4.0K Oct 30 10:48 ..
drwx------  2 root root 4.0K Oct 26 15:10 gguf
drwx------  2 root root 4.0K Oct 24 02:27 nemotron_bolt_e4b_fineweb_topo_hf
drwx------  2 root root 4.0K Oct 24 01:07 nemotron_bolt_e4b_fineweb_topo
drwx------  2 root root 4.0K Oct 19 19:52 hf_cache
drwx------  2 root root 4.0K Oct 15 10:02 deepseek-r1-distill-llama-70b-topocbp-nolora
drwx------  2 root root 4.0K Oct 15 01:11 mixtral-8x22b-topocbp-nolora
drwx------  2 root root 4.0K Oct 12 20:02 riemann_mendeleev_70b_run
drwx------  2 root root 4.0K Oct 12 14:15 llama_70b_rh_demo_run
drwx------  2 root root 4.0K Oct 11 11:17 gpt_oss_20b_rh_demo_run
drwx------  2 root root 4.0K Oct  9 10:47 gpt_oss_20b_hf_topo_run


In [2]:
!rm -rf /content/drive/MyDrive/topogpt/muse-glimmer-30b-topo-cbp-fineweb-Q4_K_M.gguf

In [ ]:
!ls -ltha

In [ ]:
import os, subprocess
subprocess.run("pip install -q -U --force-reinstall transformers tokenizers huggingface_hub safetensors accelerate", shell=True, check=True)
os.kill(os.getpid(), 9)

In [ ]:
import os, sys, subprocess
from pathlib import Path

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

# ---- one-time setup (runs once, then restarts the runtime) ----
if not os.path.exists("deps_v2.flag"):
    sh("pip install -q -U --force-reinstall transformers tokenizers huggingface_hub safetensors accelerate")
    sh("pip install -q -U sentencepiece protobuf numpy")
    sh("test -d llama.cpp || git clone --depth 1 https://github.com/ggml-org/llama.cpp")
    sh("pip install -q --no-deps ./llama.cpp/gguf-py")   # NOT llama.cpp/requirements.txt
    sh("cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF > /dev/null")
    sh("cmake --build llama.cpp/build --target llama-quantize -j > /dev/null")
    open("deps_v2.flag", "w").write("ok")
    print("Setup done. Runtime is restarting: run this cell again.")
    os.kill(os.getpid(), 9)

In [1]:
import subprocess
subprocess.run("cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF", shell=True, check=True)
subprocess.run("cmake --build llama.cpp/build --config Release -j", shell=True, check=True)

CompletedProcess(args='cmake --build llama.cpp/build --config Release -j', returncode=0)

## gguf

In [4]:
import gc
try:
    del model
except NameError:
    pass
gc.collect()

import sys, subprocess
from pathlib import Path

# Treat mismatched companion packages as absent so transformers skips their imports
import transformers
import transformers.utils as _tu
import transformers.utils.import_utils as _iu
_SKIP = ("is_torchvision", "is_torchaudio", "is_torchcodec")
for _m in (_tu, _iu):
    for _n in dir(_m):
        if _n.startswith(_SKIP):
            setattr(_m, _n, lambda *a, **k: False)

import hashlib
import torch
import torch.nn as nn
from transformers import AutoModelForMultimodalLM, AutoTokenizer

REPO_ID = "frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb"
BASE_MODEL_ID = "meta-models/Muse-Glimmer-30B"
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
QUANT = "Q4_K_M"          # or Q5_K_M, Q8_0
OUT_NAME = "/content/drive/MyDrive/topogpt/muse-glimmer-30b-card-faithful"

llama = Path("llama.cpp")
work = Path("muse_gguf_work"); work.mkdir(exist_ok=True)
card_dir = work / "card_faithful_hf"

assert Path("/content/drive/MyDrive").exists(), "Google Drive is not mounted."
Path(OUT_NAME).parent.mkdir(parents=True, exist_ok=True)
assert (llama / "convert_hf_to_gguf.py").exists(), "llama.cpp is missing from the working directory."
assert (llama / "build/bin/llama-quantize").exists(), "llama-quantize is not built."

# 1. Tokenizer from the governed repository (as in the card)
print(f"[*] Downloading tokenizer from {REPO_ID}...")
tokenizer = AutoTokenizer.from_pretrained(REPO_ID)
print("tokenizer length:", len(tokenizer))

# 2. Base model with the card's class, bf16 and unquantized (GGUF needs real weights).
#    As in the card's code, lm_head.pt is NOT applied.
print(f"[*] Loading base model {BASE_MODEL_ID} (bf16, CPU)...")
model = AutoModelForMultimodalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
model.eval()

# 3. Card's silicon audit on the input embeddings
if hasattr(model, "get_input_embeddings"):
    embedding_layer = model.get_input_embeddings()
elif hasattr(model, "model") and hasattr(model.model, "embed_tokens"):
    embedding_layer = model.model.embed_tokens
else:
    embedding_layer = next(m for m in model.modules() if isinstance(m, nn.Embedding))

weight_data = embedding_layer.weight.data
valid_anchors = [idx for idx in PRIME_ANCHORS if idx < weight_data.shape[0]]
serialized = torch.cat([weight_data[idx].flatten().float() for idx in valid_anchors]).cpu().numpy().tobytes()
audit_hash = hashlib.sha256(serialized).hexdigest()
print(f"[Audit] Anchors: {valid_anchors}")
print(f"[Audit] Silicon Hardware Digest: {audit_hash}")
print("[Audit] matches card:", audit_hash.startswith("c8b99e403ffc1fb4"))

# 4. Save the base model + the governed repo's tokenizer (+ processor if available)
model.save_pretrained(card_dir, safe_serialization=True)
tokenizer.save_pretrained(card_dir)
try:
    from transformers import AutoProcessor
    AutoProcessor.from_pretrained(BASE_MODEL_ID).save_pretrained(card_dir)
except Exception as e:
    print("[i] no processor saved:", type(e).__name__)
del model

# 5. Convert to GGUF (bf16 intermediate stays local, not on Drive)
bf16_gguf = work / "muse-bf16.gguf"
subprocess.run([sys.executable, str(llama / "convert_hf_to_gguf.py"), str(card_dir),
                "--outfile", str(bf16_gguf), "--outtype", "bf16"], check=True)

# 6. Quantize straight to Drive
quant_gguf = Path(f"{OUT_NAME}-{QUANT}.gguf")
subprocess.run([str(llama / "build/bin/llama-quantize"), str(bf16_gguf), str(quant_gguf), QUANT], check=True)
print("Wrote:", quant_gguf)

# 7. Card's generation prompt, plain completion (no chat template)
prompt = "The fundamental challenge of continual learning in neural architectures is"
r = subprocess.run(
    [str(llama / "build/bin/llama-completion"), "-m", str(quant_gguf), "-p", prompt, "-no-cnv",
     "-n", "50", "--temp", "0.7", "--top-p", "0.9", "--repeat-penalty", "1.15"],
    capture_output=True, text=True, stdin=subprocess.DEVNULL, timeout=3600,
)
print("exit code:", r.returncode)
print(r.stdout[-1500:])

[*] Downloading tokenizer from frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb...


tokenizer_config.json:   0%|          | 0.00/80.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 28.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/9.99k [00:00<?, ?B/s]

tokenizer length: 202048
[*] Loading base model meta-models/Muse-Glimmer-30B (bf16, CPU)...


Loading weights:   0%|          | 0/1436 [00:00<?, ?it/s]

[Audit] Anchors: [2, 3, 5, 7, 11, 13]
[Audit] Silicon Hardware Digest: c8b99e403ffc1fb48069aa0db3365713e70d1224fb2c052effe436b04c5c54c1
[Audit] matches card: True


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

[i] no processor saved: ModuleNotFoundError
Wrote: /content/drive/MyDrive/topogpt/muse-glimmer-30b-card-faithful-Q4_K_M.gguf
exit code: 0
The fundamental challenge of continual learning in neural architectures is not merely technical but philosophical: how can a system preserve the structure of past experience while remaining plastic enough to accommodate novel information? The paper's central contribution lies in recognizing that this tension manifests most acutely at the level of gradient dynamics, where the




## hf

In [6]:
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi

# Retrieve your token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

api = HfApi(token=HF_TOKEN)

REPO = "frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb-gguf"
GGUF = "/content/drive/MyDrive/topogpt/muse-glimmer-30b-card-faithful-Q4_K_M.gguf"
assert Path(GGUF).exists(), "GGUF not found on Drive."

README = "\n".join([
    "---",
    "base_model: meta-models/Muse-Glimmer-30B",
    "tags:",
    "- gguf",
    "- llama.cpp",
    "license: other",
    "---",
    "",
    "# Muse-Glimmer-30B Topo-Governed CBP (FineWeb) GGUF (Q4_K_M)",
    "",
    "GGUF conversion of `meta-models/Muse-Glimmer-30B`, quantized to Q4_K_M with llama.cpp.",
    "",
    "## What this is",
    "- Weights: the unmodified base model (bf16 converted to GGUF, then quantized to Q4_K_M).",
    "- Tokenizer files: taken from `frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb`, as in that repo's inference code.",
    "- The `lm_head.pt` from that repo is NOT applied. Its own inference code does not use it either.",
    "",
    "## Notes",
    "- The model is multimodal in the original; this GGUF is text-only.",
    "- Plain completion tested with `llama-completion -no-cnv`. Chat mode is not supported by this build.",
    "- License: follows the base model's license; check the base repo before use or redistribution.",
])

api.create_repo(REPO, repo_type="model", private=False, exist_ok=True)
api.update_repo_settings(repo_id=REPO, private=False)   # makes it public even if it was created private earlier
api.upload_file(path_or_fileobj=README.encode(), path_in_repo="README.md", repo_id=REPO)
api.upload_file(path_or_fileobj=GGUF, path_in_repo=Path(GGUF).name, repo_id=REPO)
print("Uploaded: https://huggingface.co/" + REPO)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...card-faithful-Q4_K_M.gguf:   0%|          | 31.7MB / 16.9GB            

Validating                    :           |   0%            

Uploaded: https://huggingface.co/frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb-gguf


## inference-hf

In [7]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
  --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 231.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [8]:
!pip show llama-cpp-python huggingface_hub

Name: llama_cpp_python
Version: 0.3.36
Summary: Python bindings for the llama.cpp library
Home-page: https://github.com/abetlen/llama-cpp-python
Author: 
Author-email: Andrei Betlen <abetlen@gmail.com>
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: diskcache, jinja2, numpy, typing-extensions
Required-by: 
---
Name: huggingface_hub
Version: 2.4.0
Summary: Client library to download and publish models, datasets and other repos on the huggingface.co hub
Home-page: https://github.com/huggingface/huggingface_hub
Author: Hugging Face, Inc.
Author-email: julien@huggingface.co
License: Apache-2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: click, filelock, fsspec, hf-xet, httpx2, packaging, pyyaml, tqdm, typing-extensions
Required-by: accelerate, datasets, diffusers, gradio, gradio_client, peft, sentence-transformers, timm, tokenizers, torchtune, transformers


In [2]:
# pip install llama-cpp-python
# (GPU build: CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --no-cache-dir)

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

REPO = "frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb-gguf"
FILE = "muse-glimmer-30b-card-faithful-Q4_K_M.gguf"

# 1. Download the GGUF from the Hub
print(f"[*] Downloading {FILE} from {REPO}...")
gguf_path = hf_hub_download(repo_id=REPO, filename=FILE)

# 2. Load the model (n_gpu_layers=-1 offloads everything if built with CUDA; ignored on CPU builds)
llm = Llama(
    model_path=gguf_path,
    n_ctx=2048,
    n_gpu_layers=-1,
    verbose=False,
)

# 3. Plain completion, same prompt and sampling settings as the model card's test
prompt = "The fundamental challenge of continual learning in neural architectures is"
out = llm.create_completion(
    prompt,
    max_tokens=512,
    temperature=0.7,
    top_p=0.9,
    repeat_penalty=1.15,
)

print(f"Prompt: {prompt}")
print(f"Generated: {out['choices'][0]['text'].strip()}")

[*] Downloading muse-glimmer-30b-card-faithful-Q4_K_M.gguf from frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb-gguf...


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


Prompt: The fundamental challenge of continual learning in neural architectures is
Generated: the catastrophic forgetting phenomenon, where learning new tasks erases knowledge of previous ones. Recent work proposes a novel solution through adaptive sparse subnetworks that dynamically allocate parameters per task while maintaining fixed overall network capacity. The core conclusion is that this approach achieves near-perfect retention of prior task performance (over 99% accuracy) with minimal parameter overhead (~1-2%), significantly outperforming traditional methods like fine-tuning, regularization-based approaches such as EWC and SI, replay buffers, and even other sparse subnetwork baselines.

A critical yet subtle constraint underpins this success: the **sparsity ratio of new subnetworks must be set to approximately 0.02 (2% of parameters)** for each task after the initial task. This specific value is not arbitrary but represents a carefully calibrated trade-off between plasticity an